# 03 · Real-world benchmark

Notebook 02 showed the test set can't be trusted: it shares the training data's blind spots, so model D scored best there while flagging Gmail and Google Docs. From now on, models get judged on data from **outside** PhiUSIIL:

| source | label | what it tests |
|---|---|---|
| Tranco top-1M list | legit | real domains, many TLDs, half from the long tail |
| your Chrome history | legit | real URLs with subdomains and paths |
| OpenPhish feed (today) | phishing | current attacks, not 2023 ones |

Two numbers matter most: **false alarm rate** on real legit URLs (would users tolerate it?) and **recall** on fresh phishing (does it still catch attacks?).

**Privacy:** your browser history is personal and may include work or internal sites. Notebook outputs get committed to GitHub, so this notebook only prints *aggregate* numbers for history unless you set `SHOW_PRIVATE = True`. If you do, clear that cell's output before committing.

**Safety:** OpenPhish URLs are live phishing pages. They're only handled as text here. Never click them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

from phishdetect.data import load_phiusiil, PROJECT_ROOT
from phishdetect.features import host_only, featurize
from phishdetect.models import build_models
from phishdetect.benchmark import load_tranco, load_openphish, load_chrome_history

pd.set_option("display.max_colwidth", 100)
RANDOM_STATE = 42
THRESHOLD = 0.5
SHOW_PRIVATE = False   # True prints individual history URLs. Clear outputs before committing if so.

## 1. Train the four models
Same data and same split as notebook 02. The models now live in `src/phishdetect/models.py`, so this notebook and the next one build them identically. Each model takes raw URLs and does its own cleanup.

The test-set PR-AUC printed here should match notebook 02. If it doesn't, something changed and the comparison isn't fair.

In [ ]:
%%time
df = load_phiusiil().drop_duplicates("url").reset_index(drop=True)
y = df["is_phishing"].values
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
urls = df["url"].values

models = build_models(RANDOM_STATE)
for name, m in models.items():
    m.fit(urls[idx_train], y[idx_train])
    print(f"{name:24s} test-set PR-AUC = {average_precision_score(y[idx_test], m.phish_proba(urls[idx_test])):.4f}")

## 2. Load the benchmark
The first run downloads Tranco (about 10 MB) and today's OpenPhish feed into `data/raw/`. OpenPhish files are saved with the date, so rerunning on later days builds a history for drift checks.

If a download fails (a firewall, an SSL error), download the file in your browser and save it as `data/raw/tranco_top1m.csv.zip` or `data/raw/openphish_YYYY-MM-DD.txt` using today's date.

In [ ]:
tranco  = load_tranco(n=5000, random_state=RANDOM_STATE)
phish   = load_openphish()
history = load_chrome_history(n=1000, max_per_host=3, random_state=RANDOM_STATE)

bench = pd.concat([tranco, history, phish], ignore_index=True)
bench["is_phishing"] = bench["is_phishing"].astype(int)
bench["host"] = bench["url"].map(host_only)
bench.groupby("source").size().rename("urls")

In [ ]:
# Public sources only. History stays private.
display(tranco.sample(5, random_state=1)[["url", "rank"]])
display(phish.sample(min(5, len(phish)), random_state=1))

## 3. Overlap with the training data
If `google.com` is already in the training set, getting it right is memory, not skill. Mark every benchmark URL whose hostname appeared in training, and report results on **unseen** hosts separately.

In [ ]:
train_hosts = set(df["url"].iloc[idx_train].map(host_only))
bench["seen_in_train"] = bench["host"].isin(train_hosts)
bench.groupby("source")["seen_in_train"].mean().round(3).rename("share seen in training")

## 4. The scoreboard
For each model, at a threshold of 0.5:

- **FA rate**: share of legit URLs wrongly flagged (lower is better).
- **recall**: share of phishing caught (higher is better).
- **bench PR-AUC**: ranking quality over the whole benchmark. The class mix here is arbitrary (it depends on how many URLs each source gave), so use it to compare models with each other, not as an absolute score.

In [ ]:
def rate(mask, flagged):
    return flagged[mask].mean() if mask.any() else np.nan

src, unseen = bench["source"], ~bench["seen_in_train"]
rows = []
for name, m in models.items():
    p = m.phish_proba(bench["url"].values)
    bench[f"p_{name[0]}"] = p
    flagged = pd.Series(p >= THRESHOLD, index=bench.index)
    rows.append({
        "model": name,
        "Tranco FA": rate(src == "tranco", flagged),
        "Tranco FA (unseen)": rate((src == "tranco") & unseen, flagged),
        "history FA": rate(src == "history", flagged),
        "OpenPhish recall": rate(src == "openphish", flagged),
        "OpenPhish recall (unseen)": rate((src == "openphish") & unseen, flagged),
        "bench PR-AUC": average_precision_score(bench["is_phishing"], p),
    })
scoreboard = pd.DataFrame(rows).set_index("model").round(3)
scoreboard

In [ ]:
ax = scoreboard[["Tranco FA", "history FA", "OpenPhish recall"]].plot.barh(figsize=(8, 4.5))
ax.set_xlim(0, 1); ax.set_xlabel("rate (FA: lower is better, recall: higher is better)")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

## 5. Testing the subdomain theory
Notebook 02's guess was that legit training hosts never have subdomains, so the models learned "subdomain means phishing." If that's right, history URLs **with** a subdomain (`docs.google.com`) should get flagged much more often than ones without (`github.com`).

`n_subdomains` counts labels beyond the last two, so `bbc.co.uk` counts as one subdomain. That's a known rough edge, but good enough to check the pattern.

In [ ]:
legit = bench[bench["is_phishing"] == 0].copy()
legit["has_subdomain"] = featurize(legit["host"])["n_subdomains"].values > 0
fa = {name: (legit[f"p_{name[0]}"] >= THRESHOLD) for name in models}
pd.DataFrame({name: f.groupby([legit["source"], legit["has_subdomain"]]).mean() for name, f in fa.items()}).round(3)

## 6. Errors
Public sources only. These are the cases to learn from.

In [ ]:
for col in ["p_C", "p_D"]:
    print(f"Tranco domains most confidently flagged by {col[-1]}:")
    display(bench[(src == "tranco")].nlargest(12, col)[["url", "rank", "seen_in_train", col]])

In [ ]:
print("OpenPhish URLs model D was most sure were safe (as text, don't open):")
display(bench[src == "openphish"].nsmallest(15, "p_D")[["url", "seen_in_train", "p_D"]])

In [ ]:
hist_rows = bench[src == "history"]
if SHOW_PRIVATE:
    display(hist_rows.nlargest(20, "p_D")[["host", "p_C", "p_D"]])
else:
    print(f"{len(hist_rows)} history URLs; model D flags {(hist_rows['p_D'] >= THRESHOLD).sum()}.")
    print("Set SHOW_PRIVATE = True to see which hosts (and clear the output before committing).")

## 7. Drift: has phishing moved on?
Model D leaned hard on Freenom's free TLDs (`.tk .ml .ga .cf .gq`). Freenom stopped free registrations in 2023. Compare TLD and hosting-platform shares in the training phishing vs today's feed.

In [ ]:
FREENOM = {"tk", "ml", "ga", "cf", "gq"}
PLATFORMS = ("web.app", "firebaseapp.com", "pages.dev", "workers.dev", "vercel.app", "netlify.app",
             "github.io", "glitch.me", "ipfs", "weebly", "wixsite.com", "blogspot.", "r2.dev")

def profile(hosts):
    tld = hosts.str.rsplit(".", n=1).str[-1]
    return {
        "Freenom TLD share": tld.isin(FREENOM).mean(),
        "free hosting platform share": hosts.apply(lambda h: any(p in h for p in PLATFORMS)).mean(),
        "top TLDs": ", ".join(tld.value_counts().head(8).index),
    }

train_phish_hosts = df.loc[idx_train][df.loc[idx_train, "is_phishing"] == 1]["url"].map(host_only)
pd.DataFrame({"training phishing": profile(train_phish_hosts),
              "OpenPhish today": profile(bench.loc[src == "openphish", "host"])})

## 8. Save the scoreboard
Rates only, no URLs, so it's safe to commit. Each future notebook adds a row, so you can see whether changes actually help.

In [ ]:
from datetime import date
out = PROJECT_ROOT / "reports"
out.mkdir(exist_ok=True)
sb = scoreboard.assign(notebook="03", date=date.today().isoformat(),
                       n_tranco=(src == "tranco").sum(), n_history=(src == "history").sum(),
                       n_openphish=(src == "openphish").sum())
sb.to_csv(out / "scoreboard_03.csv")
sb

## Results log
Fill in after running, and copy into the journal.

| model | Tranco FA | history FA | OpenPhish recall | bench PR-AUC |
|---|---|---|---|---|
| A · original | | | | |
| B · no scheme | | | | |
| C · host only | | | | |
| D · host char n-grams | | | | |

Questions to answer in the journal:
1. Which model has the best balance of false alarms and recall on *real* data? Is it the one that won on the test set?
2. Did the subdomain theory hold up (section 5)?
3. Has the phishing TLD mix shifted since the training data (section 7)?

**Before committing:** if you set `SHOW_PRIVATE = True`, clear outputs first.